# 🏛️ A.R.C. — Adaptive Reasoning Chain
**AIMO Progress Prize 3 Submission**

Optimized from:
- Notebook 1 (44/50): Infrastructure & architecture
- Notebook 2 V40 (42/50): Hyperparameters & prompts
- GenSelect: NVIDIA AIMO-2 winning selection technique
- 5-component weighted entropy: Proven +5-6 pts over simple mean

In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(input_archive, temp_dir):
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-index',
        '--find-links', f'{temp_dir}/wheels',
        'unsloth', 'trl', 'vllm', 'openai_harmony'
    ], check=True)

In [ ]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz',
    temp_dir='/kaggle/tmp/setup'
)

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [ ]:
import warnings
warnings.simplefilter('ignore')

import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, load_harmony_encoding,
    SystemContent, ReasoningEffort, ToolNamespaceConfig,
    Author, Message, Role, TextContent, Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

## Configuration
V40-optimized: simple prompts (+9pts), temp=1.0 (+8pts), 65K context (+6pts), no top_p (+8pts)

In [ ]:
class CFG:
    # === PROMPTS: V40-style simple (42/50) NOT verbose (33/50) ===
    system_prompt = (
        'You are a world-class IMO competitor. '
        'The final answer must be 0-99999. '
        'Place answer inside \\boxed{}.'
    )
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Brute-force verification for small cases\n\n'
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results.\n\n'
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you are computing and why before running code.'
    )
    preference_prompt = (
        'You have access to math, numpy, sympy, and z3 (constraint solver) for:\n\n'
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation, solving equations, number theory\n'
        '- Polynomial factorization, modular arithmetic\n\n'
        '# Numerical Computation (numpy):\n'
        '- Array operations, linear algebra, matrix operations\n\n'
        '# Constraint Solving (z3):\n'
        '- Integer constraints, satisfiability problems\n\n'
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically'
    )

    # === MODEL ===
    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'

    # === INFERENCE ===
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'
    gpu_memory_utilization = 0.96
    context_tokens = 65536      # NOT 131K (would regress -6)
    temperature = 1.0           # NOT 0.5 (would regress -8)
    min_p = 0.02
    # NO top_p — omitted entirely (top_p=0.9 regresses -8)

    # === ROLLOUTS ===
    high_problem_timeout = 900
    base_problem_timeout = 300
    notebook_limit = 17400
    server_timeout = 180
    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3
    stream_interval = 200
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    # === GENSELECT ===
    genselect_enabled = True
    genselect_temperature = 0.1
    genselect_max_tokens = 512
    genselect_trace_limit = 4000
    genselect_top_k = 3

set_seed(CFG.seed)

## Chat Template

In [ ]:
class AIMO3Template:

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:
        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, system_prompt: str, user_prompt: str, tool_config: ToolNamespaceConfig
    ) -> list[Message]:
        system_content = self.get_system_content(system_prompt, tool_config)
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)
        user_message = Message.from_role_and_content(Role.USER, user_prompt)
        return [system_message, user_message]

## Sandbox (Persistent Jupyter Kernels)

In [ ]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    # Pre-import preamble — includes z3 for constraint problems
    PREAMBLE = (
        'import math\n'
        'import numpy\n'
        'import sympy\n'
        'import itertools\n'
        'import collections\n'
        'import mpmath\n'
        'mpmath.mp.dps = 64\n'
        'try:\n'
        '    from z3 import *\n'
        'except ImportError:\n'
        '    pass\n'
    )

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout: float):
        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None

        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(self.PREAMBLE)

    def _format_error(self, traceback: list[str]) -> str:
        clean_lines = []
        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)
            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue
            clean_lines.append(clean_frame)
        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:
        client = self._client
        effective_timeout = timeout or self._default_timeout

        msg_id = client.execute(
            code, store_history=True, allow_stdin=False, stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time
            if elapsed > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')
                if content.get('name') == 'stdout':
                    stdout_parts.append(text)
                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])
                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')
                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr
        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def reset(self):
        self.execute('%reset -f\n' + self.PREAMBLE)

    def close(self):
        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()
        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def __del__(self):
        self.close()

## Tool Bridge (Code Execution)

In [ ]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):
        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        self._owns_session = sandbox is None
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:
        lines = code.strip().split('\n')
        if not lines:
            return code
        last_line = lines[-1].strip()
        if 'print' in last_line or 'import' in last_line:
            return code
        if not last_line or last_line.startswith('#'):
            return code
        lines[-1] = 'print(' + last_line + ')'
        return '\n'.join(lines)

    @property
    def instruction(self) -> str:
        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:
        return ToolNamespaceConfig(name='python', description=self.instruction, tools=[])

    def _make_response(self, output: str, channel: str | None = None) -> Message:
        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')
        if channel:
            message = message.with_channel(channel)
        return message

    def process_sync_plus(self, message: Message) -> list[Message]:
        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)
            except TimeoutError as exc:
                output = f'[ERROR] {exc}'
        return [self._make_response(output, channel=message.channel)]

## 5-Component Weighted Entropy
Proven +5-6 pts over simple mean entropy (V111: 40 vs V136: 35)

In [ ]:
def compute_weighted_entropy(logprobs_buffer: list) -> float:
    """5-component weighted entropy — the proven winning formula.
    
    Components:
      1. Base mean entropy (30%)
      2. Std dev penalty (20%) — penalize inconsistent confidence
      3. Position-weighted entropy (40%) — recent tokens matter more
      4. High entropy penalty (30% * 3.0) — penalize uncertain stretches
      5. Low entropy streak bonus (-10%) — reward consistent confidence
    """
    if not logprobs_buffer:
        return float('inf')

    entropies = []
    for top_logprobs_dict in logprobs_buffer:
        if not isinstance(top_logprobs_dict, dict) or not top_logprobs_dict:
            continue
        token_entropy = 0.0
        for token_str, log_prob in top_logprobs_dict.items():
            prob = math.exp(log_prob)
            if prob > 0:
                token_entropy -= prob * math.log2(prob)
        entropies.append(token_entropy)

    if not entropies:
        return float('inf')

    n = len(entropies)

    # 1. Base mean entropy
    mean_ent = sum(entropies) / n

    # 2. Variance penalty
    variance = sum((e - mean_ent) ** 2 for e in entropies) / n
    std_dev = math.sqrt(variance)

    # 3. Position-weighted (most important — 40%)
    decay_factor = 0.995
    weights = [decay_factor ** (n - 1 - i) for i in range(n)]
    total_weight = sum(weights)
    position_weighted = sum(w * e for w, e in zip(weights, entropies)) / total_weight

    # 4. High entropy penalty
    high_ent_count = sum(1 for e in entropies if e > 2.0)
    high_ent_ratio = high_ent_count / n

    # 5. Low entropy streak bonus
    max_streak = 0
    current_streak = 0
    for e in entropies:
        if e < 1.0:
            current_streak += 1
            max_streak = max(max_streak, current_streak)
        else:
            current_streak = 0
    streak_bonus = -0.1 * (max_streak / n)

    return (
        0.3 * mean_ent +
        0.4 * position_weighted +
        0.2 * std_dev +
        0.3 * high_ent_ratio * 3.0 +
        streak_bonus
    )

## Answer Extraction Utilities

In [ ]:
def scan_for_answer(text: str) -> int | None:
    """Extract answer from \\boxed{} or 'final answer is' patterns."""
    # Primary: \boxed{N}
    pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
    matches = re.findall(pattern, text)
    if matches:
        try:
            value = int(matches[-1].replace(',', ''))
            if 0 <= value <= 99999:
                return value
        except ValueError:
            pass

    # Fallback: 'final answer is N'
    pattern = r'final\s+answer\s+is\s*([0-9,]+)'
    matches = re.findall(pattern, text, re.IGNORECASE)
    if matches:
        try:
            value = int(matches[-1].replace(',', ''))
            if 0 <= value <= 99999:
                return value
        except ValueError:
            pass

    return None

## GenSelect + Hybrid Answer Selection

In [ ]:
class GenSelector:
    """Generative Best-of-N selection (NVIDIA AIMO-2 technique).
    
    Instead of just counting votes, reads full reasoning traces
    and selects the most mathematically rigorous answer.
    """

    def __init__(self, client, cfg, encoding):
        self.client = client
        self.cfg = cfg
        self.encoding = encoding

    def select(self, problem: str, candidates: list[dict]) -> int | None:
        groups = defaultdict(list)
        for c in candidates:
            if c.get('Answer') is not None:
                groups[c['Answer']].append(c)

        if len(groups) <= 1:
            return list(groups.keys())[0] if groups else None

        # Rank by total entropy weight, take top-K
        ranked = sorted(
            groups.items(),
            key=lambda x: sum(1.0 / max(c.get('Entropy', float('inf')), 1e-9) for c in x[1]),
            reverse=True
        )[:self.cfg.genselect_top_k]

        prompt = self._build_prompt(problem, ranked)

        try:
            response = self.client.completions.create(
                model=self.cfg.served_model_name,
                prompt=prompt,
                temperature=self.cfg.genselect_temperature,
                max_tokens=self.cfg.genselect_max_tokens,
                extra_body={'min_p': self.cfg.min_p}
            )
            text = response.choices[0].text
            selected = scan_for_answer(text)
            if selected is not None and selected in groups:
                return selected
        except Exception:
            pass

        return None  # fallback to entropy

    def _build_prompt(self, problem, ranked_groups):
        parts = [
            f'Problem: {problem}\n\n',
            'Multiple solution attempts produced different answers. ',
            'Analyze each reasoning trace and select the most ',
            'mathematically rigorous answer.\n\n'
        ]
        for i, (answer, traces) in enumerate(ranked_groups):
            best = min(traces, key=lambda t: t.get('Entropy', float('inf')))
            trace_text = best.get('Trace', '')[:self.cfg.genselect_trace_limit]
            parts.append(f'=== Solution {i+1} (Answer: {answer}, Votes: {len(traces)}) ===\n')
            parts.append(f'{trace_text}\n\n')
        parts.append(
            'Which solution is most mathematically rigorous? '
            'Reply with the correct answer inside \\boxed{}.'
        )
        return ''.join(parts)


def select_answer(detailed_results: list, problem: str = '', gen_selector=None) -> int:
    """Hybrid selection: consensus → GenSelect → 5-component entropy."""
    answer_weights = defaultdict(float)
    answer_votes = defaultdict(int)

    for result in detailed_results:
        answer = result.get('Answer')
        entropy = result.get('Entropy', float('inf'))
        if answer is not None:
            weight = 1.0 / max(entropy, 1e-9)
            answer_weights[answer] += weight
            answer_votes[answer] += 1

    if not answer_weights:
        return 0

    # Stage 1: Strong consensus (4+ agree) — submit immediately
    top_answer = max(answer_votes, key=answer_votes.get)
    if answer_votes[top_answer] >= 4:
        print(f'  [CONSENSUS] {top_answer} with {answer_votes[top_answer]} votes')
        return top_answer

    # Stage 2: GenSelect for divergent answers
    if gen_selector and len(answer_votes) >= 2:
        try:
            gs_answer = gen_selector.select(problem, detailed_results)
            if gs_answer is not None:
                print(f'  [GENSELECT] Selected {gs_answer}')
                return gs_answer
        except Exception as e:
            print(f'  [GENSELECT] Failed: {e}')

    # Stage 3: Weighted entropy fallback
    scored = sorted(
        [{'answer': a, 'votes': answer_votes[a], 'score': s}
         for a, s in answer_weights.items()],
        key=lambda x: x['score'], reverse=True
    )

    vote_df = pd.DataFrame(scored)
    vote_df = vote_df.round({'score': 3})
    display(vote_df)

    winner = scored[0]['answer']
    print(f'  [ENTROPY] Selected {winner}')
    return winner

## Main Solver

In [ ]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()

        self._preload_model_weights()
        self.server_process = self._start_server()

        self.client = OpenAI(
            base_url=self.base_url, api_key=self.api_key,
            timeout=self.cfg.session_timeout
        )

        self._wait_for_server()
        self._initialize_kernels()

        # GenSelect selector
        self.gen_selector = None
        if self.cfg.genselect_enabled:
            self.gen_selector = GenSelector(self.client, self.cfg, self.encoding)

        self.notebook_start_time = time.time()
        self.problems_remaining = 50

    def _preload_model_weights(self) -> None:
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        files_to_load = []
        total_size = 0
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:
            with open(path, 'rb') as f:
                while f.read(1024 * 1024 * 1024):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f}s\n')

    def _start_server(self) -> subprocess.Popen:
        cmd = [
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
            '--seed', str(self.cfg.seed),
            '--model', self.cfg.model_path,
            '--served-model-name', self.cfg.served_model_name,
            '--tensor-parallel-size', '1',
            '--max-num-seqs', str(self.cfg.batch_size),
            '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization),
            '--host', '0.0.0.0', '--port', str(self.port),
            '--dtype', self.cfg.dtype,
            '--kv-cache-dtype', self.cfg.kv_cache_dtype,
            '--max-model-len', str(self.cfg.context_tokens),
            '--stream-interval', str(self.cfg.stream_interval),
            '--async-scheduling', '--disable-log-stats',
            '--enable-prefix-caching'
        ]
        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(
            cmd, stdout=self.log_file, stderr=subprocess.STDOUT,
            start_new_session=True
        )

    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start_time = time.time()
        for _ in range(self.cfg.server_timeout):
            rc = self.server_process.poll()
            if rc is not None:
                self.log_file.flush()
                with open('vllm_server.log', 'r') as f:
                    logs = f.read()
                raise RuntimeError(f'Server died with code {rc}. Logs:\n{logs}\n')
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f}s).\n')
                return
            except Exception:
                time.sleep(1)
        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self) -> None:
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
        self.sandbox_pool = queue.Queue()
        def _create():
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create) for _ in range(self.cfg.workers)]
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
        print(f'Kernels initialized in {time.time() - start_time:.2f}s.\n')

    def _process_attempt(
        self, problem: str, system_prompt: str,
        attempt_index: int, stop_event: threading.Event, deadline: float
    ) -> dict:
        if stop_event.is_set() or time.time() > deadline:
            return {'Attempt': attempt_index + 1, 'Answer': None,
                    'Python Calls': 0, 'Python Errors': 0,
                    'Response Length': 0, 'Entropy': float('inf'), 'Trace': ''}

        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        logprobs_buffer = []
        all_text_chunks = []
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout,
                tool_prompt=self.cfg.tool_prompt, sandbox=sandbox
            )
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, problem, local_tool.tool_config
            )
            conversation = Conversation.from_messages(messages)

            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
                if max_tokens < self.cfg.buffer_tokens:
                    break

                stream = self.client.completions.create(
                    model=self.cfg.served_model_name,
                    temperature=self.cfg.temperature,
                    logprobs=self.cfg.top_logprobs,
                    max_tokens=max_tokens,
                    prompt=prompt_ids,
                    seed=attempt_seed,
                    stream=True,
                    extra_body={
                        'min_p': self.cfg.min_p,
                        'stop_token_ids': self.stop_token_ids,
                        'return_token_ids': True
                    }
                )

                try:
                    token_buffer = []
                    text_chunks = []

                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            all_text_chunks.append(new_text)
                            chunk_logprobs = chunk.choices[0].logprobs
                            if chunk_logprobs is not None and chunk_logprobs.top_logprobs:
                                logprobs_buffer.extend(chunk_logprobs.top_logprobs)

                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = scan_for_answer(search_text)
                            if answer is not None:
                                final_answer = answer
                                break
                finally:
                    stream.close()

                if final_answer is not None:
                    break
                if not token_buffer:
                    break

                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = scan_for_answer(answer_text)
                    break

                if last_message.recipient == 'python':
                    python_calls += 1
                    tool_responses = local_tool.process_sync_plus(last_message)
                    response_text = tool_responses[0].content[0].text
                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1
                    conversation.messages.extend(tool_responses)

        except Exception:
            python_errors += 1
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        entropy = compute_weighted_entropy(logprobs_buffer)

        return {
            'Attempt': attempt_index + 1,
            'Response Length': total_tokens,
            'Python Calls': python_calls,
            'Python Errors': python_errors,
            'Entropy': entropy,
            'Answer': final_answer,
            'Trace': ''.join(all_text_chunks[-200:])  # Last ~200 chunks for GenSelect
        }

    def solve_problem(self, problem: str) -> int:
        print(f'\nProblem: {problem}\n')
        user_input = f'{problem} {self.cfg.preference_prompt}'

        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        reserved_time = max(0, self.problems_remaining - 1) * self.cfg.base_problem_timeout
        budget = min(max(time_left - reserved_time, self.cfg.base_problem_timeout), self.cfg.high_problem_timeout)
        deadline = time.time() + budget

        print(f'Budget: {budget:.2f}s | Problems remaining: {self.problems_remaining}\n')

        detailed_results = []
        valid_answers = []
        stop_event = threading.Event()
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = [
                executor.submit(
                    self._process_attempt, user_input,
                    self.cfg.system_prompt, i, stop_event, deadline
                )
                for i in range(self.cfg.attempts)
            ]

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
                    counts = Counter(valid_answers).most_common(1)
                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()
                        for f in futures:
                            f.cancel()
                        break
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)

        if detailed_results:
            df = pd.DataFrame(detailed_results)
            df['Entropy'] = df['Entropy'].round(3)
            df['Answer'] = df['Answer'].astype('Int64')
            display(df[['Attempt', 'Response Length', 'Python Calls', 'Python Errors', 'Entropy', 'Answer']])

        if not valid_answers:
            print('\nResult: 0\n')
            return 0

        final = select_answer(detailed_results, problem, self.gen_selector)
        print(f'\nFinal Answer: {final}\n')
        return final

    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
        if hasattr(self, 'log_file'):
            self.log_file.close()
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    self.sandbox_pool.get_nowait().close()
                except Exception:
                    pass

## Initialize & Run

In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    id_value = id_.item(0)
    question_text = question.item(0)
    gc.disable()
    final_answer = solver.solve_problem(question_text)
    gc.enable()
    gc.collect()
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )